# Chapter 15 — When Training Goes Wrong

**Book alignment:** Debugging AI From First Principles, Chapter 15

**Question this notebook isolates:** A flat loss curve at `ln(10) ≈ 2.303`. Triage order
H3 → H2 → H1: does interrogating the instrumentation (raw vs. reported), then the overfit-64
probe (data vs. optimizer), then one LR change — each with a pre-written curve prediction —
convict exactly one system without changing all three at once?

In [ ]:
import numpy as np

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def ce(logits, y):
    p = softmax(logits)
    return float(-np.log(p[np.arange(len(y)), y] + 1e-12).mean())

rng = np.random.default_rng(0)
N, D, K = 256, 8, 10
X = rng.normal(0, 1, (N, D))
w_true = rng.normal(0, 1, (D, K))
y = (X @ w_true).argmax(1)                       # a learnable linear problem

def train(steps, lr, *, labels=None, w0=None):
    yy = y if labels is None else labels
    w = np.zeros((D, K)) if w0 is None else w0.copy()
    losses, gnorms = [], []
    for _ in range(steps):
        logits = X @ w
        p = softmax(logits)
        oh = np.eye(K)[yy]
        g = X.T @ (p - oh) / N
        w -= lr * g
        losses.append(ce(X @ w, yy)); gnorms.append(float(np.linalg.norm(g)))
    return w, losses, gnorms

## 1. Loss at init is a fact, not a hope: `ln(K)`

In [ ]:
init = ce(np.zeros((N, K)), y)
print(f"loss at init: {init:.3f}   ln(10) = {np.log(10):.3f}")
assert abs(init - np.log(10)) < 1e-6
print("a curve stuck HERE is chance-level - three systems could hold it there")

## 2. H3 first — interrogate the instrumentation at zero training cost

In [ ]:
w, raw_losses, _ = train(60, lr=5e-1)
# the logging bug: the 'reported' curve recomputes loss from the INITIAL weights every epoch
reported = [ce(np.zeros((N, K)), y) for _ in range(60)]
print(f"raw    per-step loss: {raw_losses[0]:.3f} -> {raw_losses[-1]:.3f}")
print(f"reported curve      : {reported[0]:.3f} -> {reported[-1]:.3f}")
assert raw_losses[-1] < raw_losses[0] - 0.5       # training IS descending
assert abs(reported[-1] - reported[0]) < 1e-9     # the plot is flat
print("raw moves, reported does not -> H3: the curve is journalism, not training. no GPU owed.")

## 3. The overfit-64 probe: data (H2) vs. optimizer (H1)

In [ ]:
sub = slice(0, 64)
Xs = X[sub]

def overfit(labels, lr, steps=400):
    w = np.zeros((D, K))
    for _ in range(steps):
        p = softmax(Xs @ w); oh = np.eye(K)[labels]
        g = Xs.T @ (p - oh) / 64
        w -= lr * g
    return ce(Xs @ w, labels), float(np.linalg.norm(g))

loss_ok, gn_ok = overfit(y[sub], lr=5e-1)
loss_shuf, gn_shuf = overfit(rng.permutation(y[sub]), lr=5e-1)
gn_blocked = 0.0                                  # an H1 flow-block would show ~0 gradients from step 0
print(f"real labels    : loss {loss_ok:.3f}   grad-norm {gn_ok:.4f}")
print(f"shuffled labels: loss {loss_shuf:.3f}   grad-norm {gn_shuf:.4f}")
assert loss_ok < 0.3                              # healthy pipeline memorises the subset
assert loss_shuf > 1.0                            # cannot memorise noise -> H2 (data), not LR
assert gn_shuf > gn_blocked                       # ... and gradients DID flow (not an H1 flow-block)
print("fails to overfit WITH gradients flowing -> data carries no signal (H2), not an LR problem")

## 4. Only now: sweep one optimizer variable

In [ ]:
_, lo_lo, _ = train(60, lr=2e-3)                  # LR too low
_, lo_hi, _ = train(60, lr=5e-1)                  # LR ×~250
print(f"LR 2e-3 : {lo_lo[0]:.3f} -> {lo_lo[-1]:.3f}   (barely moves)")
print(f"LR 5e-1 : {lo_hi[0]:.3f} -> {lo_hi[-1]:.3f}   (descends)")
assert lo_lo[-1] > lo_lo[0] - 0.3                 # flat
assert lo_hi[-1] < lo_hi[0] - 1.0                 # responsive
print("curve slope tracks LR monotonically -> H1-LR, and ONLY after H3/H2 were exonerated")

## What we earned

Training is three coupled systems — instrumentation, data stream, optimizer loop — and the
curve is the output of the first describing the third fed by the second. Triage runs
H3 → H2 → H1: the raw-vs-reported check convicted a logging bug with zero GPU cost; the
overfit-64 probe (with the label-shuffle control) separated "no signal" from "no descent";
the LR sweep came last and only after the cheap systems were cleared. `ln(K)` at init is the
number every classification curve starts from.

**Notebook 16 / Chapter 16** meets the honestly-trained model with an excellent score — and
puts the *evaluation* on trial.